# 🟨 Capa Gold: Modelado Analítico, Conversión Monetaria y Agregaciones de Negocio
Este notebook unifica los mercados europeos. Aplica la conversión monetaria de Zlotys (PLN) a Euros (\$€\$) para Polonia, reduce su granularidad de 15 minutos a 1 hora mediante promedios, genera los KPIs diarios agregados y exporta snapshots planos para la portabilidad del entorno local.

In [13]:
import pyspark.sql.functions as F

print("🚀 Iniciando unificación y reglas de negocio de la Capa Gold...")

FACTOR_CONVERSION_PLN_A_EUR = 0.23  # Tasa fija EUR/PLN — actualizar si cambia la política de conversión

df_de = spark.read.table("silver_precios_alemania")
df_es = spark.read.table("silver_precios_espana")
df_ro = spark.read.table("silver_precios_rumania")
df_pl = spark.read.table("silver_precios_polonia")

# Polonia: conversión PLN→EUR y reducción PT15M→PT60M por promedio horario
df_pl_eur = df_pl.withColumn("precio_mwh", F.col("precio_mwh") * FACTOR_CONVERSION_PLN_A_EUR)

df_pl_horario = df_pl_eur.withColumn("timestamp_utc", F.date_trunc("hour", F.col("timestamp_utc"))) \
                         .groupBy("timestamp_utc", "pais", "fuente") \
                         .agg(F.round(F.avg("precio_mwh"), 2).alias("precio_mwh")) \
                         .withColumn("timestamp_utc", F.col("timestamp_utc").cast("string"))

StatementMeta(, e82c62cc-92d4-43bb-b5e2-81bed948be40, 15, Finished, Available, Finished, False)

🚀 Iniciando unificación y reglas de negocio de la Capa Gold...


In [14]:
def normalizar_esquema(df):
    return df.select(
        F.col("timestamp_utc").cast("string").alias("timestamp_utc"),
        F.col("precio_mwh").cast("double").alias("precio_mwh"),
        F.col("pais").cast("string").alias("pais"),
        F.col("fuente").cast("string").alias("fuente")
    )

df_de_norm = normalizar_esquema(df_de)
df_es_norm = normalizar_esquema(df_es)
df_ro_norm = normalizar_esquema(df_ro)
df_pl_norm = normalizar_esquema(df_pl_horario)

df_gold_horario = df_de_norm.unionByName(df_es_norm) \
                            .unionByName(df_ro_norm) \
                            .unionByName(df_pl_norm)

df_gold_horario.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_precios_horarios_europa")
print("💾 Tabla Delta Horaria 'gold_precios_horarios_europa' creada con éxito.")

StatementMeta(, e82c62cc-92d4-43bb-b5e2-81bed948be40, 16, Finished, Available, Finished, False)

💾 Tabla Delta Horaria 'gold_precios_horarios_europa' creada con éxito.


In [15]:
print("📊 Generando agregación analítica diaria...")

df_diario_prep = df_gold_horario.withColumn("fecha", F.to_date(F.col("timestamp_utc")))

df_gold_diario = df_diario_prep.groupBy("fecha", "pais", "fuente").agg(
    F.round(F.avg("precio_mwh"), 2).alias("precio_promedio_eur"),
    F.round(F.max("precio_mwh"), 2).alias("precio_maximo_eur"),
    F.round(F.min("precio_mwh"), 2).alias("precio_minimo_eur"),
    F.count("precio_mwh").alias("total_muestras_horarias")
)

df_gold_diario.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_precios_diarios_agregados")

print("📤 Exportando snapshots en archivos CSV para portabilidad...")

df_gold_horario.coalesce(1).write \
               .mode("overwrite") \
               .option("header", "true") \
               .csv("Files/gold_precios_horarios_export")

df_gold_diario.coalesce(1).write \
              .mode("overwrite") \
              .option("header", "true") \
              .csv("Files/gold_precios_diarios_export")

print("✅ Proceso de datos finalizado con éxito.")

StatementMeta(, e82c62cc-92d4-43bb-b5e2-81bed948be40, 17, Finished, Available, Finished, False)

📊 Generando agregación analítica diaria...
📤 Exportando snapshots en archivos CSV para portabilidad...
✅ Proceso de datos finalizado con éxito.
